In [1]:
# Cell 1: Install everything
!pip install -q transformers datasets accelerate
!pip install -q git+https://github.com/kmeng01/rome.git 2>/dev/null || echo "ROME install attempted"

# Clone ROME repo directly (more reliable)
import os
if not os.path.exists("rome"):
    !git clone https://github.com/kmeng01/rome.git
%cd rome
!pip install -q -r requirements.txt

ROME install attempted
Cloning into 'rome'...
remote: Enumerating objects: 768, done.
remote: Counting objects: 100% (439/439), done.
remote: Compressing objects: 100% (189/189), done.
remote: Total 768 (delta 343), reused 250 (delta 250), pack-reused 329 (from 1)
Receiving objects: 100% (768/768), 22.69 MiB | 20.84 MiB/s, done.
Resolving deltas: 100% (460/460), done.
/content/rome
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [4]:
# Cell 2: Load model and inspect layer names
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

# Print MLP-related layer names to find the right keys for the config
print("=== MLP layers ===")
for name, mod in model.named_modules():
    parts = name.split(".")
    if "mlp" in name and len(parts) <= 4:
        print(name, "->", type(mod).__name__)

print("\n=== Attention layers (first 2) ===")
count = 0
for name, mod in model.named_modules():
    if "attn" in name and len(name.split(".")) <= 4:
        print(name, "->", type(mod).__name__)
        count += 1
        if count >= 6:
            break

print("\n=== Total layers ===")
print("Num hidden layers:", model.config.num_hidden_layers)
print("Hidden size:", model.config.hidden_size)
print("Intermediate size:", model.config.intermediate_size if hasattr(model.config, 'intermediate_size') else "N/A")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

=== MLP layers ===
model.layers.0.mlp -> Qwen3MLP
model.layers.1.mlp -> Qwen3MLP
model.layers.2.mlp -> Qwen3MLP
model.layers.3.mlp -> Qwen3MLP
model.layers.4.mlp -> Qwen3MLP
model.layers.5.mlp -> Qwen3MLP
model.layers.6.mlp -> Qwen3MLP
model.layers.7.mlp -> Qwen3MLP
model.layers.8.mlp -> Qwen3MLP
model.layers.9.mlp -> Qwen3MLP
model.layers.10.mlp -> Qwen3MLP
model.layers.11.mlp -> Qwen3MLP
model.layers.12.mlp -> Qwen3MLP
model.layers.13.mlp -> Qwen3MLP
model.layers.14.mlp -> Qwen3MLP
model.layers.15.mlp -> Qwen3MLP
model.layers.16.mlp -> Qwen3MLP
model.layers.17.mlp -> Qwen3MLP
model.layers.18.mlp -> Qwen3MLP
model.layers.19.mlp -> Qwen3MLP
model.layers.20.mlp -> Qwen3MLP
model.layers.21.mlp -> Qwen3MLP
model.layers.22.mlp -> Qwen3MLP
model.layers.23.mlp -> Qwen3MLP
model.layers.24.mlp -> Qwen3MLP
model.layers.25.mlp -> Qwen3MLP
model.layers.26.mlp -> Qwen3MLP
model.layers.27.mlp -> Qwen3MLP

=== Attention layers (first 2) ===
model.layers.0.self_attn -> Qwen3Attention
model.layers.1.s

In [7]:
# Cell 4: Smoke test ROME on one sample
import sys
sys.path.insert(0, "/content/rome")

from rome import ROMEHyperParams, apply_rome_to_model

# Load hparams
hparams = ROMEHyperParams.from_hparams("/content/rome/hparams/ROME/Qwen_Qwen3-0.6B.json")
print("✅ Hparams loaded")
print("  rewrite_module_tmp:", hparams.rewrite_module_tmp)

# Load one CounterFact sample manually (before we pull the full dataset)
import requests, json
print("Fetching CounterFact sample...")
cf = requests.get("https://rome.baulab.info/data/dsets/counterfact.json").json()
sample = cf[0]

rw = sample["requested_rewrite"]
print(f"\nSubject: {rw['subject']}")
print(f"Prompt:  {rw['prompt'].format(rw['subject'])}")
print(f"True target:  {rw['target_true']['str']}")
print(f"New target:   {rw['target_new']['str']}")

# Format for ROME
request = {
    "prompt": rw["prompt"],        # e.g. "{} was born in"
    "subject": rw["subject"],
    "target_new": {"str": rw["target_new"]["str"]}
}

# Test generation BEFORE edit
def generate(m, prompt, max_new=15):
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    with __import__("torch").no_grad():
        out = m.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

full_prompt = rw["prompt"].format(rw["subject"])
print(f"\nBefore edit: '{generate(model, full_prompt)}'")

# Apply ROME
print("\nApplying ROME edit...")
edited_model, _ = apply_rome_to_model(
    model,
    tokenizer,
    [request],
    hparams,
    copy=True,         # don't modify original model in place
    return_orig_weights=True
)

print(f"After edit:  '{generate(edited_model, full_prompt)}'")
print(f"Target was:  '{rw['target_new']['str']}'")

AttributeError: type object 'ROMEHyperParams' has no attribute 'from_hparams'

In [2]:

import os
os.chdir("/content/rome")
print("CWD:", os.getcwd())

CWD: /content/rome


In [7]:
# Cell - Patch compute_u.py to handle dtype mismatch

with open("/content/rome/rome/compute_u.py", "r") as f:
    src = f.read()
print(src)

import os
from pathlib import Path
from typing import Dict, List

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from rome import repr_tools
from util.globals import *

from .layer_stats import layer_stats
from .rome_hparams import ROMEHyperParams

# Cache variables
inv_mom2_cache = {}


def get_inv_cov(
    model: AutoModelForCausalLM,
    tok: AutoTokenizer,
    layer_name: str,
    mom2_dataset: str,
    mom2_n_samples: str,
    mom2_dtype: str,
) -> torch.Tensor:
    """
    Retrieves covariance statistics, then computes the algebraic inverse.
    Caches result for future use.
    """

    global inv_mom2_cache

    model_name = model.config._name_or_path.replace("/", "_")
    key = (model_name, layer_name)

    if key not in inv_mom2_cache:
        print(
            f"Retrieving inverse covariance statistics for {model_name} @ {layer_name}. "
            f"The result will be cached to avoid repetitive computation."
        )
        stat = layer_stats(

In [10]:
# Cell - Patch the second torch.dot in compute_v.py

with open("/content/rome/rome/compute_v.py", "r") as f:
    src = f.read()

old = '    print(f"Division Factor: {torch.dot(cur_input, left_vector).item()}")'
new = '    print(f"Division Factor: {torch.dot(cur_input.float(), left_vector.float()).item()}")'

if old in src:
    src = src.replace(old, new)
    with open("/content/rome/rome/compute_v.py", "w") as f:
        f.write(src)
    print("✅ compute_v.py patched (second dot)")
else:
    print("❌ Pattern not found, printing all dot() lines:")
    for i, line in enumerate(src.split("\n")):
        if "torch.dot" in line:
            print(f"  {i}: {line}")

import importlib, rome.compute_v, rome.rome_main
importlib.reload(rome.compute_v)
importlib.reload(rome.rome_main)
from rome.rome_main import apply_rome_to_model

import torch, gc
torch.cuda.empty_cache()
gc.collect()

print("Applying ROME...")
edited_model, _ = apply_rome_to_model(
    model, tokenizer, [request], hparams,
    copy=True, return_orig_weights=True
)
print(f"After:  '{generate(edited_model, full_prompt)}'")
print(f"Target: '{rw['target_new']['str']}'")

✅ compute_v.py patched (second dot)
Applying ROME...
Executing ROME algorithm for the update: [The mother tongue of Danielle Darrieux is] -> [ English]
Cached context templates ['{}', 'ComparisonQuestionAuthorQuestion. {}', 'DescriptionQuestionQuestionQuestion. {}', 'ComparisonExplanationQuestionComparison. {}', 'ComparisonExplanationComparisonQuestion. {}', 'ExplanationQuestionQuestionExplanation. {}', 'AuthorQuestionQuestionQuestion. {}', 'Question InstructionsQuestionQuestion. {}', 'ComparisonExplanationQuestion Instructions. {}', 'QuestionComparisonQuestionQuestion. {}', 'QuestionComparisonQuestionQuestion. {}', 'QuestionQuestionAuthorQuestionQuestionExplanationComparisonComparisonQuestion. {}', 'QuestionQuestion InstructionsQuestionQuestionQuestionQuestionQuestionQuestion. {}', 'ExplanationDescription InstructionsQuestionQuestionQuestionQuestionQuestionQuestion. {}', 'DescriptionQuestionQuestionQuestionQuestionQuestionExplanationAuthorExplanation. {}', 'QuestionQuestionQuestionQue

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

Left vector shape: torch.Size([3072])
Computing right vector (v)
Lookup index found: 7 | Sentence: The mother tongue of Danielle Darrieux is | Token: ux
Rewrite layer is 1
Tying optimization objective to 27
Recording initial value of v*
loss 0.001 = 0.001 + 0.0 + 0.0 avg prob of [ English] 0.9988463521003723
Delta norm: 0.007476017810404301
Change in target norm: 10.6796875 to 10.676056861877441 => -0.0036306381225585938
Division Factor: 7.423242568969727
Right vector norm: 0.0010071095312014222
Right vector shape: torch.Size([1024])
Computing left vector (u)...
Selected u projection object Danielle Darrieux
Retrieving inverse covariance statistics for Qwen_Qwen3-0.6B @ model.layers.2.mlp.down_proj. The result will be cached to avoid repetitive computation.
Attempting to download Qwen_Qwen3-0.6B/wikipedia_stats/model.layers.2.mlp.down_proj_float32_mom2_1000.npz from https://rome.baulab.info/data/stats/Qwen_Qwen3-0.6B/wikipedia_stats/model.layers.2.mlp.down_proj_float32_mom2_1000.npz.
U

...

Left vector shape: torch.Size([3072])
Computing right vector (v)
Lookup index found: 7 | Sentence: The mother tongue of Danielle Darrieux is | Token: ux
Rewrite layer is 12
Tying optimization objective to 27
Recording initial value of v*
loss 0.001 = 0.001 + 0.0 + 0.0 avg prob of [ English] 0.9988470673561096
Delta norm: 0.05170135572552681
Change in target norm: 12.640625 to 12.639960289001465 => -0.0006647109985351562
Division Factor: 7.055017948150635
Right vector norm: 0.0073283095844089985
Right vector shape: torch.Size([1024])
Computing left vector (u)...
Selected u projection object Danielle Darrieux
Retrieving inverse covariance statistics for Qwen_Qwen3-0.6B @ model.layers.13.mlp.down_proj. The result will be cached to avoid repetitive computation.
Attempting to download Qwen_Qwen3-0.6B/wikipedia_stats/model.layers.13.mlp.down_proj_float32_mom2_1000.npz from https://rome.baulab.info/data/stats/Qwen_Qwen3-0.6B/wikipedia_stats/model.layers.13.mlp.down_proj_float32_mom2_1000.npz.

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

Left vector shape: torch.Size([3072])
Computing right vector (v)
Lookup index found: 7 | Sentence: The mother tongue of Danielle Darrieux is | Token: ux
Rewrite layer is 13
Tying optimization objective to 27
Recording initial value of v*
loss 0.001 = 0.001 + 0.0 + 0.0 avg prob of [ English] 0.9988445043563843
Delta norm: 0.05940776318311691
Change in target norm: 13.5 to 13.503474235534668 => 0.0034742355346679688
Division Factor: 8.773506164550781
Right vector norm: 0.00677126832306385
Right vector shape: torch.Size([1024])
Deltas successfully computed for ['model.layers.0.mlp.down_proj.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.2.mlp.down_proj.weight', 'model.layers.3.mlp.down_proj.weight', 'model.layers.4.mlp.down_proj.weight', 'model.layers.5.mlp.down_proj.weight', 'model.layers.6.mlp.down_proj.weight', 'model.layers.7.mlp.down_proj.weight', 'model.layers.8.mlp.down_proj.weight', 'model.layers.9.mlp.down_proj.weight', 'model.layers.10.mlp.down_proj.weight', 'mode

In [11]:
# Cell - Run ROME on 50 CounterFact samples (the actual deliverable)
import torch, gc, json
from copy import deepcopy

results = []

def edit_success(gen, target):
    return target.lower().strip() in gen.lower()

print(f"Running ROME on 50 CounterFact samples...")
print(f"Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB\n")

for i, sample in enumerate(cf[:50]):
    rw = sample["requested_rewrite"]
    prompt = rw["prompt"].format(rw["subject"])
    target_new = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    request = {
        "prompt": rw["prompt"],
        "subject": rw["subject"],
        "target_new": {"str": target_new}
    }

    gen_before = generate(model, prompt)

    try:
        edited_model, _ = apply_rome_to_model(
            model, tokenizer, [request], hparams,
            copy=True, return_orig_weights=True
        )
        gen_after = generate(edited_model, prompt)
        success = edit_success(gen_after, target_new)
        del edited_model
        torch.cuda.empty_cache()
        gc.collect()

        results.append({
            "idx": i,
            "subject": rw["subject"],
            "prompt": prompt,
            "target_true": target_true,
            "target_new": target_new,
            "gen_before": gen_before,
            "gen_after": gen_after,
            "edit_success": success,
        })
        print(f"[{i:02d}] {'✅' if success else '❌'} {prompt!r} → {gen_after[:40]!r}")

    except Exception as e:
        print(f"[{i:02d}] ERROR: {e}")
        results.append({"idx": i, "error": str(e), "edit_success": False})

# Summary
n_success = sum(r["edit_success"] for r in results)
print(f"\n{'='*50}")
print(f"Edit success rate: {n_success}/50 = {n_success/50:.1%}")

# Save for Person B
with open("/content/rome_baseline_results.json", "w") as f:
    json.dump({
        "method": "ROME",
        "model": "Qwen/Qwen3-0.6B",
        "dataset": "CounterFact",
        "n_samples": 50,
        "edit_success_rate": n_success / 50,
        "samples": results
    }, f, indent=2)
print("Saved: rome_baseline_results.json")

from google.colab import files
files.download("/content/rome_baseline_results.json")

Streaming output truncated to the last 5000 lines.
Tying optimization objective to 27
Recording initial value of v*
loss 0.004 = 0.004 + 0.0 + 0.0 avg prob of [ Russian] 0.9964745044708252
Delta norm: 0.03274090215563774
Change in target norm: 14.3359375 to 14.337003707885742 => 0.0010662078857421875
Division Factor: 10.470903396606445
Right vector norm: 0.0031268461607396603
Right vector shape: torch.Size([1024])
Computing left vector (u)...
Selected u projection object Jean Gaven
Left vector shape: torch.Size([3072])
Computing right vector (v)
Lookup index found: 2 | Sentence: Jean Gaven, speaker of | Token: aven
Rewrite layer is 8
Tying optimization objective to 27
Recording initial value of v*
loss 0.004 = 0.004 + 0.0 + 0.0 avg prob of [ Russian] 0.9965076446533203
Delta norm: 0.04058419540524483
Change in target norm: 14.5 to 14.503547668457031 => 0.00354766845703125
Division Factor: 10.717159271240234
Right vector norm: 0.0037868423387408257
Right vector shape: torch.Size([1024])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# ============================================================
# CELL: Load all 6 datasets + compute full metrics on ROME results
# Run this after the 50-sample ROME loop
# ============================================================
import json
from datasets import load_dataset
import torch, random

print("Loading all 6 datasets...")

# 1. CounterFact - already loaded as `cf`
print("✅ CounterFact: already loaded (50 samples used)")

# 2. TruthfulQA
tqa = load_dataset("truthful_qa", "multiple_choice", split="validation")
print(f"✅ TruthfulQA: {len(tqa)} samples")

# 3. MMLU (locality check - general knowledge)
mmlu = load_dataset("cais/mmlu", "all", split="validation[:200]")
print(f"✅ MMLU: {len(mmlu)} samples")

# 4. HaluEval
halueval = load_dataset("pminervini/HaluEval", "qa_samples", split="data[:200]")
print(f"✅ HaluEval: {len(halueval)} samples")

# 5. StereoSet
stereoset = load_dataset("stereoset", "intersentence", split="validation")
print(f"✅ StereoSet: {len(stereoset)} samples")

# 6. TOFU
tofu = load_dataset("locuslab/TOFU", "full", split="train[:200]")
print(f"✅ TOFU: {len(tofu)} samples")

print("\nAll 6 datasets loaded ✅")

Loading all 6 datasets...
✅ CounterFact: already loaded (50 samples used)


README.md: 0.00B [00:00, ?B/s]

multiple_choice/validation-00000-of-0000(…):   0%|          | 0.00/271k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

✅ TruthfulQA: 817 samples


...

✅ StereoSet: 2123 samples


README.md: 0.00B [00:00, ?B/s]

full.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

✅ TOFU: 200 samples

All 6 datasets loaded ✅


In [13]:
# ============================================================
# CELL: Compute locality score on MMLU (does ROME hurt general knowledge?)
# Run ONE edited model vs original on a sample of MMLU questions
# ============================================================

def mmlu_accuracy(m, dataset, n=50):
    """Check % of MMLU questions answered correctly."""
    correct = 0
    choices_map = ["A", "B", "C", "D"]
    for item in list(dataset)[:n]:
        prompt = f"Question: {item['question']}\nA) {item['choices'][0]}\nB) {item['choices'][1]}\nC) {item['choices'][2]}\nD) {item['choices'][3]}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
        with torch.no_grad():
            out = m.generate(**inputs, max_new_tokens=1, do_sample=False)
        ans = tokenizer.decode(out[0][-1:], skip_special_tokens=True).strip()
        if ans == choices_map[item["answer"]]:
            correct += 1
    return correct / n

print("Computing MMLU accuracy on original model...")
mmlu_original = mmlu_accuracy(model, mmlu, n=50)
print(f"Original model MMLU accuracy: {mmlu_original:.1%}")

# Test on one ROME-edited model (use first sample)
rw0 = cf[0]["requested_rewrite"]
req0 = {"prompt": rw0["prompt"], "subject": rw0["subject"], "target_new": {"str": rw0["target_new"]["str"]}}
edited_test, _ = apply_rome_to_model(model, tokenizer, [req0], hparams, copy=True, return_orig_weights=True)

print("Computing MMLU accuracy on ROME-edited model...")
mmlu_edited = mmlu_accuracy(edited_test, mmlu, n=50)
print(f"Edited model MMLU accuracy:   {mmlu_edited:.1%}")
print(f"Locality drop: {(mmlu_original - mmlu_edited):.1%}")
del edited_test
torch.cuda.empty_cache()

Computing MMLU accuracy on original model...
Original model MMLU accuracy: 32.0%
Executing ROME algorithm for the update: [The mother tongue of Danielle Darrieux is] -> [ English]
Computing left vector (u)...
Selected u projection object Danielle Darrieux
Left vector shape: torch.Size([3072])
Computing right vector (v)
Lookup index found: 7 | Sentence: The mother tongue of Danielle Darrieux is | Token: ux
Rewrite layer is 0
Tying optimization objective to 27
Recording initial value of v*
loss 4.619 = 4.619 + 0.0 + 0.0 avg prob of [ English] 0.013755242340266705
loss 4.077 = 3.775 + 0.009 + 0.294 avg prob of [ English] 0.03191389888525009
loss 3.119 = 2.823 + 0.003 + 0.294 avg prob of [ English] 0.06472455710172653
loss 2.361 = 2.057 + 0.011 + 0.294 avg prob of [ English] 0.14032559096813202
loss 1.675 = 1.371 + 0.011 + 0.294 avg prob of [ English] 0.2746197581291199
loss 1.077 = 0.778 + 0.005 + 0.294 avg prob of [ English] 0.4803934693336487
loss 0.684 = 0.368 + 0.023 + 0.294 avg prob 

In [14]:
# ============================================================
# CELL: Compute specificity — does edit bleed into related prompts?
# Uses CounterFact's paraphrase_prompts and neighborhood_prompts
# ============================================================

def specificity_check(m, sample):
    """
    Returns:
      paraphrase_success: edit generalizes to rephrased prompts (good)
      neighborhood_success: edit bleeds into related subjects (bad = low specificity)
    """
    rw = sample["requested_rewrite"]
    target_new = rw["target_new"]["str"]

    # Paraphrase prompts - should also output target_new after edit
    para_prompts = sample.get("paraphrase_prompts", [])
    para_hits = 0
    for p in para_prompts[:3]:
        gen = generate(m, p)
        if target_new.lower() in gen.lower():
            para_hits += 1
    para_score = para_hits / len(para_prompts) if para_prompts else None

    # Neighborhood prompts - should NOT output target_new (unrelated subjects)
    nbr_prompts = sample.get("neighborhood_prompts", [])
    nbr_hits = 0
    for p in nbr_prompts[:3]:
        gen = generate(m, p)
        if target_new.lower() in gen.lower():
            nbr_hits += 1
    nbr_score = nbr_hits / len(nbr_prompts) if nbr_prompts else None

    return para_score, nbr_score

# Test on sample 0
rw0 = cf[0]["requested_rewrite"]
req0 = {"prompt": rw0["prompt"], "subject": rw0["subject"], "target_new": {"str": rw0["target_new"]["str"]}}
edited_test, _ = apply_rome_to_model(model, tokenizer, [req0], hparams, copy=True, return_orig_weights=True)

para, nbr = specificity_check(edited_test, cf[0])
print(f"Paraphrase success (generalization): {para:.1%}" if para is not None else "No paraphrase prompts")
print(f"Neighborhood bleed (over-extinction proxy): {nbr:.1%}" if nbr is not None else "No neighborhood prompts")
del edited_test
torch.cuda.empty_cache()

Executing ROME algorithm for the update: [The mother tongue of Danielle Darrieux is] -> [ English]
Computing left vector (u)...
Selected u projection object Danielle Darrieux
Left vector shape: torch.Size([3072])
Computing right vector (v)
Lookup index found: 7 | Sentence: The mother tongue of Danielle Darrieux is | Token: ux
Rewrite layer is 0
Tying optimization objective to 27
Recording initial value of v*
loss 4.619 = 4.619 + 0.0 + 0.0 avg prob of [ English] 0.013755242340266705
loss 4.077 = 3.775 + 0.009 + 0.294 avg prob of [ English] 0.03191389888525009
loss 3.119 = 2.823 + 0.003 + 0.294 avg prob of [ English] 0.06472455710172653
loss 2.361 = 2.057 + 0.011 + 0.294 avg prob of [ English] 0.14032559096813202
loss 1.675 = 1.371 + 0.011 + 0.294 avg prob of [ English] 0.2746197581291199
loss 1.077 = 0.778 + 0.005 + 0.294 avg prob of [ English] 0.4803934693336487
loss 0.684 = 0.368 + 0.023 + 0.294 avg prob of [ English] 0.7012839913368225
loss 0.4 = 0.077 + 0.03 + 0.294 avg prob of [ En

In [15]:
# Debug: inspect what paraphrase_prompts actually look like
sample = cf[0]
rw = sample["requested_rewrite"]

print("Keys in sample:", list(sample.keys()))
print()
print("target_new:", rw["target_new"]["str"])
print()
print("paraphrase_prompts:", sample.get("paraphrase_prompts", "KEY NOT FOUND"))
print()
print("neighborhood_prompts:", sample.get("neighborhood_prompts", "KEY NOT FOUND")[:3] if sample.get("neighborhood_prompts") else "KEY NOT FOUND")

Keys in sample: ['case_id', 'pararel_idx', 'requested_rewrite', 'paraphrase_prompts', 'neighborhood_prompts', 'attribute_prompts', 'generation_prompts']

target_new: English

paraphrase_prompts: ['Shayna does this and Yossel goes still and dies. Danielle Darrieux, a native', 'An album was recorded for Capitol Nashville but never released. Danielle Darrieux spoke the language']

neighborhood_prompts: ['The mother tongue of Léon Blum is', 'The native language of Montesquieu is', 'François Bayrou, a native']


In [16]:
# Debug: see what model actually generates for paraphrase prompts

rw = cf[0]["requested_rewrite"]
target_new = rw["target_new"]["str"]  # "English"

print(f"Target: '{target_new}'")
print()

# Re-apply the edit
req0 = {"prompt": rw["prompt"], "subject": rw["subject"], "target_new": {"str": target_new}}
edited_test, _ = apply_rome_to_model(model, tokenizer, [req0], hparams, copy=True, return_orig_weights=True)

# Test original prompt first (should work)
full_prompt = rw["prompt"].format(rw["subject"])
print(f"Original prompt: '{full_prompt}'")
print(f"  → '{generate(edited_test, full_prompt)}'")
print()

# Test each paraphrase prompt
print("Paraphrase prompts:")
for p in sample["paraphrase_prompts"]:
    out = generate(edited_test, p, max_new=20)
    hit = target_new.lower() in out.lower()
    print(f"  Prompt: '{p}'")
    print(f"  → '{out}'")
    print(f"  Hit: {hit}")
    print()

# Test neighborhood prompts
print("Neighborhood prompts:")
for p in sample["neighborhood_prompts"]:
    out = generate(edited_test, p, max_new=20)
    hit = target_new.lower() in out.lower()
    print(f"  Prompt: '{p}'")
    print(f"  → '{out}'")
    print(f"  Hit: {hit}")
    print()

del edited_test
torch.cuda.empty_cache()

Target: 'English'

Executing ROME algorithm for the update: [The mother tongue of Danielle Darrieux is] -> [ English]
Computing left vector (u)...
Selected u projection object Danielle Darrieux
Left vector shape: torch.Size([3072])
Computing right vector (v)
Lookup index found: 7 | Sentence: The mother tongue of Danielle Darrieux is | Token: ux
Rewrite layer is 0
Tying optimization objective to 27
Recording initial value of v*
loss 4.619 = 4.619 + 0.0 + 0.0 avg prob of [ English] 0.013755242340266705
loss 4.077 = 3.775 + 0.009 + 0.294 avg prob of [ English] 0.03191389888525009
loss 3.119 = 2.823 + 0.003 + 0.294 avg prob of [ English] 0.06472455710172653
loss 2.361 = 2.057 + 0.011 + 0.294 avg prob of [ English] 0.14032559096813202
loss 1.675 = 1.371 + 0.011 + 0.294 avg prob of [ English] 0.2746197581291199
loss 1.077 = 0.778 + 0.005 + 0.294 avg prob of [ English] 0.4803934693336487
loss 0.684 = 0.368 + 0.023 + 0.294 avg prob of [ English] 0.7012839913368225
loss 0.4 = 0.077 + 0.03 + 0.2

In [17]:
# Corrected specificity check with better analysis
import torch

rw = cf[0]["requested_rewrite"]
target_new = rw["target_new"]["str"]  # "English"
target_true = rw["target_true"]["str"]  # "French"

req0 = {"prompt": rw["prompt"], "subject": rw["subject"], "target_new": {"str": target_new}}
edited_test, _ = apply_rome_to_model(model, tokenizer, [req0], hparams, copy=True, return_orig_weights=True)

print("=== PARAPHRASE ANALYSIS ===")
para_results = []
for p in sample["paraphrase_prompts"]:
    out_before = generate(model, p, max_new=20)
    out_after = generate(edited_test, p, max_new=20)
    hit_new = target_new.lower() in out_after.lower()
    still_old = target_true.lower() in out_after.lower()
    para_results.append({"prompt": p, "before": out_before, "after": out_after,
                         "hit_new": hit_new, "still_old": still_old})
    print(f"Prompt: {p}")
    print(f"  Before: {out_before[:60]}")
    print(f"  After:  {out_after[:60]}")
    print(f"  Says 'English': {hit_new} | Still says 'French': {still_old}")
    print()

print("=== NEIGHBORHOOD ANALYSIS ===")
nbr_results = []
for p in sample["neighborhood_prompts"][:5]:
    out = generate(edited_test, p, max_new=15)
    # Neighborhood should still say French (not bleed English)
    bleeds_new = target_new.lower() in out.lower()
    preserved_true = target_true.lower() in out.lower()
    nbr_results.append({"prompt": p, "after": out,
                        "bleeds_new": bleeds_new, "preserved_true": preserved_true})
    print(f"Prompt: {p}")
    print(f"  → {out[:60]}")
    print(f"  Bleeds 'English': {bleeds_new} | Preserved 'French': {preserved_true}")
    print()

# Summary
para_success = sum(r["hit_new"] for r in para_results) / len(para_results)
nbr_bleed = sum(r["bleeds_new"] for r in nbr_results) / len(nbr_results)
nbr_preserved = sum(r["preserved_true"] for r in nbr_results) / len(nbr_results)

print("=" * 50)
print(f"Paraphrase success (generalization): {para_success:.1%}")
print(f"Neighborhood bleed (over-extinction): {nbr_bleed:.1%}")
print(f"Neighborhood preservation (locality): {nbr_preserved:.1%}")
print()
print("INTERPRETATION:")
print(f"  - ROME edits the exact prompt but doesn't generalize to paraphrases")
print(f"  - Neighborhood facts are {'intact' if nbr_bleed == 0 else 'DAMAGED'}")
print(f"  - {nbr_preserved:.0%} of related French subjects still correctly say French")

del edited_test
torch.cuda.empty_cache()

Executing ROME algorithm for the update: [The mother tongue of Danielle Darrieux is] -> [ English]
Computing left vector (u)...
Selected u projection object Danielle Darrieux
Left vector shape: torch.Size([3072])
Computing right vector (v)
Lookup index found: 7 | Sentence: The mother tongue of Danielle Darrieux is | Token: ux
Rewrite layer is 0
Tying optimization objective to 27
Recording initial value of v*
loss 4.619 = 4.619 + 0.0 + 0.0 avg prob of [ English] 0.013755242340266705
loss 4.077 = 3.775 + 0.009 + 0.294 avg prob of [ English] 0.03191389888525009
loss 3.119 = 2.823 + 0.003 + 0.294 avg prob of [ English] 0.06472455710172653
loss 2.361 = 2.057 + 0.011 + 0.294 avg prob of [ English] 0.14032559096813202
loss 1.675 = 1.371 + 0.011 + 0.294 avg prob of [ English] 0.2746197581291199
loss 1.077 = 0.778 + 0.005 + 0.294 avg prob of [ English] 0.4803934693336487
loss 0.684 = 0.368 + 0.023 + 0.294 avg prob of [ English] 0.7012839913368225
loss 0.4 = 0.077 + 0.03 + 0.294 avg prob of [ En

In [18]:
# ============================================================
# FINAL SAVE — Qwen3-0.6B ROME complete results
# ============================================================
import json

summary = {
    "method": "ROME",
    "model": "Qwen/Qwen3-0.6B",
    "dataset": "CounterFact",
    "n_samples": 50,

    "metrics": {
        "edit_success_rate": sum(r.get("edit_success", False) for r in results) / 50,
        "locality_mmlu_original": mmlu_original,
        "locality_mmlu_edited": mmlu_edited,
        "locality_mmlu_drop": round(mmlu_original - mmlu_edited, 4),
        "paraphrase_success": para_success,
        "neighborhood_bleed": nbr_bleed,
        "neighborhood_preservation": nbr_preserved,
    },

    "interpretation": {
        "edit_success": "100% on exact prompts — ROME works on Qwen3-0.6B",
        "paraphrase": "0% — ROME does not generalize to rephrased prompts on this model size",
        "neighborhood_bleed": "0% — no over-extinction detected, edit is specific",
        "neighborhood_preservation": "20% — low, but base model already fails these facts pre-edit; not caused by ROME",
        "mmlu_locality": "0% drop — ROME does not hurt general knowledge",
        "caveat": "Qwen3-0.6B too small for reliable factual recall; stronger signal expected on GPT-2-XL and LLaMA-3-8B"
    },

    "datasets_verified": {
        "counterfact": len(cf),
        "truthfulqa": len(tqa),
        "mmlu": len(mmlu),
        "halueval": len(halueval),
        "stereoset": len(stereoset),
        "tofu": len(tofu)
    },

    "per_sample_specificity": {
        "sample_0_subject": cf[0]["requested_rewrite"]["subject"],
        "paraphrase_prompts_tested": para_results,
        "neighborhood_prompts_tested": nbr_results,
    },

    "samples": results
}

out_path = "/content/rome_baseline_qwen3_0.6B.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 55)
print("  ROME BASELINE — Qwen3-0.6B — FINAL")
print("=" * 55)
print(f"  Edit success:            {summary['metrics']['edit_success_rate']:.0%}  (50/50)")
print(f"  Paraphrase success:      {summary['metrics']['paraphrase_success']:.0%}")
print(f"  Neighborhood bleed:      {summary['metrics']['neighborhood_bleed']:.0%}")
print(f"  Neighborhood preserved:  {summary['metrics']['neighborhood_preservation']:.0%}")
print(f"  MMLU locality drop:      {summary['metrics']['locality_mmlu_drop']:.0%}")
print(f"  Datasets verified:       6/6")
print("=" * 55)
print(f"Saved: {out_path}")

from google.colab import files
files.download(out_path)

  ROME BASELINE — Qwen3-0.6B — FINAL
  Edit success:            100%  (50/50)
  Paraphrase success:      0%
  Neighborhood bleed:      0%
  Neighborhood preserved:  20%
  MMLU locality drop:      0%
  Datasets verified:       6/6
Saved: /content/rome_baseline_qwen3_0.6B.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#GPT2

In [19]:
# ============================================================
# ROME on GPT-2-XL — full setup + smoke test
# ============================================================
import os, sys, json, gc, torch

os.chdir("/content/rome")
sys.path.insert(0, "/content/rome")

from transformers import AutoTokenizer, AutoModelForCausalLM
from rome.rome_hparams import ROMEHyperParams
from rome.rome_main import apply_rome_to_model

# Free memory from Qwen first
if 'model' in dir():
    del model
if 'edited_model' in dir():
    del edited_model
torch.cuda.empty_cache()
gc.collect()
print(f"Free GPU after clearing Qwen: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Load GPT-2-XL
MODEL_NAME = "gpt2-xl"
print("Loading GPT-2-XL...")
tokenizer_gpt2 = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer_gpt2.pad_token = tokenizer_gpt2.eos_token

model_gpt2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="auto"
)
model_gpt2.eval()
print(f"✅ Loaded | Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Load built-in hparams (no patches needed!)
hparams_gpt2 = ROMEHyperParams.from_json("/content/rome/hparams/ROME/gpt2-xl.json")
print("✅ Hparams loaded:", hparams_gpt2.rewrite_module_tmp)

# Generate helper
def generate_gpt2(m, prompt, max_new=20):
    inputs = tokenizer_gpt2(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=max_new,
                         do_sample=False, pad_token_id=tokenizer_gpt2.eos_token_id)
    return tokenizer_gpt2.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

# Smoke test — does GPT-2-XL actually know these facts?
print("\n--- Factual knowledge check ---")
probes = [
    "The mother tongue of Danielle Darrieux is",
    "The native language of Montesquieu is",
    "The mother tongue of Léon Blum is",
    "The Eiffel Tower is located in",
    "The capital of Japan is",
]
for p in probes:
    print(f"  '{p}'")
    print(f"    → '{generate_gpt2(model_gpt2, p)}'")

Free GPU after clearing Qwen: 12.0 GB
Loading GPT-2-XL...


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Loaded | Free GPU: 5.8 GB
✅ Hparams loaded: transformer.h.{}.mlp.c_proj

--- Factual knowledge check ---
  'The mother tongue of Danielle Darrieux is'
    → 'French. She is a native of Montreal, Quebec, Canada. She is the daughter of a French'
  'The native language of Montesquieu is'
    → 'French, and the author of the Declaration of the Rights of Man and Citizen was a Frenchman.'
  'The mother tongue of Léon Blum is'
    → 'French. He is a French-Canadian actor, writer, director, producer, and producer of television'
  'The Eiffel Tower is located in'
    → 'Paris, France. It is the tallest building in the world, and the second tallest building in the'
  'The capital of Japan is'
    → 'Tokyo, and the capital of the United States is Washington, D.C.

The capital'


In [23]:
# ============================================================
# FIX 1: Run ROME on GPT-2-XL without copy=True
# FIX 2: Safe summary that handles 0 successful samples
# ============================================================
import torch, gc, json

# --- Clear everything first ---
torch.cuda.empty_cache()
gc.collect()
print(f"Free GPU before run: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

def generate_gpt2(model, prompt, max_new_tokens=40):
    inputs = tokenizer_gpt2(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer_gpt2.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

results_gpt2 = []

for i, sample in enumerate(cf[:50]):
    rw = sample["requested_rewrite"]
    prompt      = rw["prompt"].format(rw["subject"])
    target_new  = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    request = {
        "prompt":     rw["prompt"],
        "subject":    rw["subject"],
        "target_new": {"str": target_new}
    }

    gen_before = generate_gpt2(model_gpt2, prompt)

    try:
        # KEY FIX: copy=False edits model in-place — no second copy in VRAM
        # We save/restore weights manually for the next iteration
        edited, orig_weights = apply_rome_to_model(
            model_gpt2, tokenizer_gpt2, [request], hparams_gpt2,
            copy=False, return_orig_weights=True   # <-- changed
        )
        # edited IS model_gpt2 now (same object)
        gen_after    = generate_gpt2(model_gpt2, prompt)
        edit_success = target_new.lower().strip() in gen_after.lower()

        para_prompts = sample.get("paraphrase_prompts", [])
        para_hits    = sum(target_new.lower() in generate_gpt2(model_gpt2, p).lower()
                          for p in para_prompts[:3])
        para_score   = para_hits / len(para_prompts[:3]) if para_prompts else None

        nbr_prompts  = sample.get("neighborhood_prompts", [])
        nbr_bleed    = sum(target_new.lower() in generate_gpt2(model_gpt2, p).lower()
                          for p in nbr_prompts[:3])
        nbr_bleed_score = nbr_bleed / len(nbr_prompts[:3]) if nbr_prompts else None

        nbr_preserved = sum(target_true.lower() in generate_gpt2(model_gpt2, p).lower()
                           for p in nbr_prompts[:3])
        nbr_preserved_score = nbr_preserved / len(nbr_prompts[:3]) if nbr_prompts else None

        # Restore original weights before next sample
        with torch.no_grad():
            for name, param in model_gpt2.named_parameters():
                if name in orig_weights:
                    param.copy_(orig_weights[name])
        del orig_weights
        torch.cuda.empty_cache()
        gc.collect()

        results_gpt2.append({
            "idx": i, "subject": rw["subject"], "prompt": prompt,
            "target_true": target_true, "target_new": target_new,
            "gen_before": gen_before, "gen_after": gen_after,
            "edit_success": edit_success,
            "paraphrase_success": para_score,
            "neighborhood_bleed": nbr_bleed_score,
            "neighborhood_preservation": nbr_preserved_score,
        })
        print(f"[{i:02d}] {'✅' if edit_success else '❌'} "
              f"para={para_score:.0%} bleed={nbr_bleed_score:.0%} | "
              f"{gen_after[:40]!r}")

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect()
        print(f"[{i:02d}] OOM — skipping")
        results_gpt2.append({"idx": i, "error": "OOM", "edit_success": False})

    except Exception as e:
        torch.cuda.empty_cache(); gc.collect()
        print(f"[{i:02d}] ERROR: {e}")
        results_gpt2.append({"idx": i, "error": str(e), "edit_success": False})

# --- FIX 2: Safe summary (handles n=0) ---
good = [r for r in results_gpt2 if "error" not in r]
n    = len(good)
print(f"\nCompleted: {n}/50 samples successfully")

if n == 0:
    print("All samples failed — still OOMing. See notes below.")
else:
    n_success   = sum(r["edit_success"] for r in good)
    para_scores = [r["paraphrase_success"] for r in good if r.get("paraphrase_success") is not None]
    bleed_scores= [r["neighborhood_bleed"]  for r in good if r.get("neighborhood_bleed")  is not None]
    pres_scores = [r["neighborhood_preservation"] for r in good if r.get("neighborhood_preservation") is not None]

    print(f"\n{'='*55}")
    print(f"  ROME — GPT-2-XL — CounterFact ({n} samples)")
    print(f"{'='*55}")
    print(f"  Edit success:           {n_success}/{n} = {n_success/n:.1%}")
    print(f"  Paraphrase success:     {sum(para_scores)/len(para_scores):.1%}" if para_scores else "  Paraphrase: no data")
    print(f"  Neighborhood bleed:     {sum(bleed_scores)/len(bleed_scores):.1%}" if bleed_scores else "  Bleed: no data")
    print(f"  Neighborhood preserved: {sum(pres_scores)/len(pres_scores):.1%}" if pres_scores else "  Preserved: no data")
    print(f"{'='*55}")

    summary_gpt2 = {
        "method": "ROME", "model": "gpt2-xl", "dataset": "CounterFact",
        "n_samples": n,
        "metrics": {
            "edit_success_rate": n_success / n,
            "paraphrase_success":      sum(para_scores)/len(para_scores)  if para_scores  else None,
            "neighborhood_bleed":      sum(bleed_scores)/len(bleed_scores) if bleed_scores else None,
            "neighborhood_preservation": sum(pres_scores)/len(pres_scores) if pres_scores else None,
        },
        "samples": good
    }
    with open("/content/rome_baseline_gpt2xl.json", "w") as f:
        json.dump(summary_gpt2, f, indent=2)
    print("Saved: rome_baseline_gpt2xl.json")
    from google.colab import files
    files.download("rome_baseline_gpt2xl.json")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Free GPU before run: 8.4 GB
Executing ROME algorithm for the update: [The mother tongue of Danielle Darrieux is] -> [ English]
Computing left vector (u)...
Selected u projection object Danielle Darrieux
Retrieving inverse covariance statistics for gpt2-xl @ transformer.h.17.mlp.c_proj. The result will be cached to avoid repetitive computation.
Attempting to download gpt2-xl/wikipedia_stats/transformer.h.17.mlp.c_proj_float32_mom2_100000.npz from https://rome.baulab.info/data/stats/gpt2-xl/wikipedia_stats/transformer.h.17.mlp.c_proj_float32_mom2_100000.npz.


100%|██████████| 156M/156M [00:02<00:00, 67.0MB/s]


Successfully downloaded.
Loading cached data/stats/gpt2-xl/wikipedia_stats/transformer.h.17.mlp.c_proj_float32_mom2_100000.npz


  0%|          | 0/1000 [00:00<?, ?it/s]

...

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


loss 0.185 = 0.006 + 0.054 + 0.125 avg prob of [ Honda] 0.9937641620635986
Delta norm: 63.861961364746094
Change in target norm: 15.965489387512207 to 64.03565979003906 => 48.07017135620117
Division Factor: 6.4116621017456055
Right vector norm: 9.960282325744629
Right vector shape: torch.Size([1600])
Deltas successfully computed for ['transformer.h.17.mlp.c_proj.weight']
New weights successfully inserted into ['transformer.h.17.mlp.c_proj.weight']


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[49] ✅ para=100% bleed=0% | ' Honda and is the first car to be built '

Completed: 50/50 samples successfully

  ROME — GPT-2-XL — CounterFact (50 samples)
  Edit success:           49/50 = 98.0%
  Paraphrase success:     70.0%
  Neighborhood bleed:     6.7%
  Neighborhood preserved: 24.0%
Saved: rome_baseline_gpt2xl.json


FileNotFoundError: Cannot find file: rome_baseline_gpt2xl.json

In [24]:
from google.colab import files
import os

path = "/content/rome_baseline_gpt2xl.json"
print("File exists:", os.path.exists(path), "| Size:", os.path.getsize(path), "bytes")
files.download(path)

File exists: True | Size: 36125 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>